In [ ]:
! aws s3 cp --recursive s3://neets/turtles/voices/donald-trump-20231114-v2/tortoise/training/prepared_clips /tmp/donald-trump-prepared-clips

In [4]:
# in "/tmp/donald-trump-prepared-clips/*.txt", replace "/tmp/tmpn5bjha_p/training/prepared_clips" with "/tmp/donald-trump-prepared-clips" using sed
! sed -i 's/\/tmp\/tmpn5bjha_p\/training\/prepared_clips/\/tmp\/donald-trump-prepared-clips/g' /tmp/donald-trump-prepared-clips/*.txt

# replace "|" with "\t"
# ! sed -i 's/|/\t/g' /tmp/donald-trump-prepared-clips/*.txt
# ! sed -i 's/\t /\t/g' /tmp/donald-trump-prepared-clips/*.txt

In [9]:
# https://github.com/neonbjb/tortoise-tts/issues/472
! pip install transformers==4.29.2
! pip install peft
! pip install peft git+https://git.ecker.tech/mrq/DL-Art-School.git

  Cloning https://git.ecker.tech/mrq/DL-Art-School.git to /tmp/pip-req-build-nl5bm_nz
  Running command git clone --filter=blob:none --quiet https://git.ecker.tech/mrq/DL-Art-School.git /tmp/pip-req-build-nl5bm_nz
  Resolved https://git.ecker.tech/mrq/DL-Art-School.git to commit a4afad8837404b6d99c0a7da0f4337da6e34fc61
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
Using legacy 'setup.py install' for dlas, since package 'wheel' is not installed.
  Running setup.py install for dlas ... done


In [1]:
import os

from huggingface_hub import hf_hub_download

from dlas.utils import options as option
from dlas.train import Trainer

autoregressive_model_path = hf_hub_download(
  "jbetker/tortoise-tts-v2",
  ".models/autoregressive.pth"
)

rel_path = "gpt_finetune.yml"
config_path = rel_path # os.path.join(os.path.abspath(''), rel_path)
opt = option.parse(config_path, is_train=True)

trainer = Trainer()
trainer.rank = -1

opt["path"]["pretrain_model_gpt"] = autoregressive_model_path
opt["path"]["experiments_root"] = os.path.join(os.path.abspath(''), "experiments")
opt["datasets"]["train"]["path"] = [
  "/tmp/donald-trump-prepared-clips/train.txt"
]
opt["datasets"]["val"]["path"] = [
  "/tmp/donald-trump-prepared-clips/validation.txt"
]

mel_norms_path = hf_hub_download(
  "jbetker/tortoise-tts-v2",
  "tortoise/data/mel_norms.pth",
)

opt["steps"]["gpt_train"]["injectors"]["paired_to_mel"]["mel_norm_file"] = mel_norms_path
opt["steps"]["gpt_train"]["injectors"]["paired_cond_to_mel"]["mel_norm_file"] = mel_norms_path
opt["steps"]["gpt_train"]["injectors"]["to_codes"]["dvae_config"] = "train_diffusion_vocoder_22k_level.yml"

launcher = "none"
mode = ""
trainer.init(config_path, opt, launcher, mode)

24-05-04 00:03:32.592 - INFO:   name: gpt
  model: extensibletrainer
  scale: 1
  gpu_ids: [0]
  start_step: 0
  checkpointing_enabled: True
  fp16: False
  bitsandbytes: True
  gpus: 1
  datasets:[
    train:[
      name: training
      n_workers: 4
      batch_size: 128
      mode: paired_voice_audio
      path: ['/tmp/donald-trump-prepared-clips/train.txt']
      fetcher_mode: ['lj']
      phase: train
      max_wav_length: 551250
      max_text_length: 350
      sample_rate: 22050
      load_conditioning: True
      num_conditioning_candidates: 3
      conditioning_length: 132300
      use_bpe_tokenizer: True
      tokenizer_vocab: bpe_lowercase_asr_256.json
      load_aligned_codes: False
      data_type: img
    ]
    val:[
      name: validation
      n_workers: 4
      batch_size: 13
      mode: paired_voice_audio
      path: ['/tmp/donald-trump-prepared-clips/validation.txt']
      fetcher_mode: ['lj']
      phase: val
      max_wav_length: 286715
      max_text_length: 200
  

Path already exists. Rename it to [/home/ubuntu/jsdir/neets-tts/packages/dlas_peft/experiments_archived_240504-000332]


24-05-04 00:03:35.780 - INFO: Number of training data elements: 1,452, iters: 12
24-05-04 00:03:35.781 - INFO: Total epochs needed: 834 for iters 10,000
/home/ubuntu/jsdir/neets-tts/.venv/lib/python3.10/site-packages/transformers/configuration_utils.py:380: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
24-05-04 00:03:45.187 - INFO: Loading model for [/home/ubuntu/.cache/huggingface/hub/models--jbetker--tortoise-tts-v2/snapshots/b20a372926b4d6132bcec0a7087f5dd14c8d9e10/.models/autoregressive.pth]


Loading from /home/ubuntu/.cache/huggingface/hub/models--jbetker--tortoise-tts-v2/snapshots/0217b1434a58714ba25187b413bc2a0a56c59fad/.models/dvae.pth


In [2]:
unified_voice = trainer.model.networks["gpt"].module

from peft import get_peft_model
from peft import LoraConfig, TaskType

gpt_model = unified_voice.gpt
#gpt_model.wte = unified_voice.text_embedding
#gpt_model.wpe = unified_voice.text_pos_embedding.emb

peft_config = LoraConfig(
  task_type=TaskType.CAUSAL_LM,
  inference_mode=False,
  r=8,
  target_modules='.*\.(4|5|6|7|8|9|10)\.mlp\.(c_proj|c_fc)',
  lora_alpha=32,
  lora_dropout=0.1,
)

peft_model = get_peft_model(gpt_model, peft_config)
peft_model.print_trainable_parameters()


/home/ubuntu/jsdir/neets-tts/.venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:1059: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


AttributeError: 'GPT2Model' object has no attribute 'wte'

In [3]:
trainer.do_training()

24-05-04 00:02:56.057 - INFO: Start training from epoch: 0, iter: 0
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [3,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [4,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1290: indexSelectLargeIndex: block: [118,0,0], thread: [5,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/nat

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
